# IBAMA licensing records

The preserved licensing snapshot becomes `ibama_licensed_projects.gpkg` with English field and category labels, preserving its records and geometry. `datasets/run.py ibama` executes these cells.

**Source:** the original extraction combined three [SIGAGEO service](https://pamgia.ibama.gov.br/server/rest/services/SIGAGEO/tb_empreendimento_repr_p/MapServer) geometry layers. The [archived extraction report](https://github.com/fab1s/brazil-project/blob/archive/full-project-2026-10-27/data/archive/original_code/Scripts_Data/IBAMA/IBAMA%20-%20Data%20Extraction.qmd) describes retrieval. This build starts with the inventoried GeoPackage; original API responses and capture time are unavailable. [Codebook](CODEBOOK.md).

In [1]:
%run ../_shared.py
ctx = notebook_context("ibama")
import pandas as pd
from IPython.display import display

## 1. Preserved source

One row is a licensing record or mapped element. Several records can relate to the same named development. The snapshot contains points, lines, and polygons in EPSG:4674. The inventory checksum prevents a newer or edited file from silently replacing this study snapshot.

In [2]:
source = ctx.path("infrastructure", "ibama")
baseline_checksum(ctx.root, source)
display(geopackage_inventory(source))
display(sql_table(source, "SELECT objectid, tp_elemento, tp_setor, tp_tipologia FROM licensed_projects_st LIMIT 5"))

,Layer,Rows,Geometry,CRS
0,licensed_projects_st,8494,Unknown,EPSG:4674


,objectid,tp_elemento,tp_setor,tp_tipologia
0,560043.0,Área de Porto,Transporte,Transporte Hidroviário Marítimo
1,560044.0,Área de Porto,Transporte,Transporte Hidroviário Marítimo
2,560045.0,Área de Porto,Transporte,Transporte Hidroviário Marítimo
3,560046.0,Área de Porto,Transporte,Transporte Hidroviário Marítimo
4,560047.0,Área de Porto,Transporte,Transporte Hidroviário Marítimo


## 2. Sector translations

The snapshot stage maps Portuguese sectors to reviewed English labels and writes a separate intermediate. It preserves source columns and geometry. The table shows the labels and counts actually found; an unknown label is a review issue. [Sector mapping](../shared/brazil_project/infrastructure/projects.py).

In [3]:
process_stage(ctx, "ibama")
from processing import verify_translation
verify_translation(ctx)
translated = pd.read_parquet(ctx.artifact("ibama", "licensed_projects.parquet"), columns=["tp_setor", "tp_sector_en"])
display(translated.groupby(["tp_setor", "tp_sector_en"], dropna=False).size().rename("Records").reset_index())

Using saved ibama outputs


,tp_setor,tp_sector_en,Records
0,Energia,Energy,2013
1,Mineração,Mining,420
2,Nuclear/Radiativa,Nuclear/Radioactive,25
3,Outros,Other,512
4,Petróleo e Gás,Oil and Gas,219
5,Recursos Hídricos,Water Resources,72
6,Transporte,Transportation,5233


## 3. English fields and category labels

The export copies the preserved GeoPackage, renames its columns, and adds English element, sector, project-type, and licence-type labels beside the original values. It preserves feature IDs and geometry blobs. Project names, developer names, numbers, and dates retain their original values. [Exact field and category mappings](../english_export.py).

In [4]:
from processing import export_geopackage, inspect_final
result = inspect_final(ctx.root, preview=True) if ctx.notebook_from_saved else export_geopackage(ctx.root)
show_final(ctx, result)
display(sql_table(result["file"], "SELECT source_field, original_value_column, english_value_column FROM field_dictionary"))
display(sql_table(result["file"], "SELECT record_id, element_type_original, element_type_english, sector_original, sector_english FROM licensed_projects_st LIMIT 5"))

,File,Layer,Features,CRS
0,datasets/ibama/ibama_licensed_projects.gpkg,licensed_projects_st,8494,EPSG:4674


SHA-256: 63f6c13fa914373cef32ec0f3073ed6a69c4eaa8c1d86d9f4b79eb8ac6ca3eb7


,source_field,original_value_column,english_value_column
0,objectid,record_id,None
1,tp_elemento,element_type_original,element_type_english
2,no_empreendimento,project_name_original,None
3,tp_setor,sector_original,sector_english
4,tp_tipologia,project_type_original,project_type_english
5,nu_sei_formatado,sei_case_number,None
6,url_licencas_emitidas,issued_licenses_url,None
7,sei_no_empreendimento,sei_project_name_original,None
8,licencas,licenses_original,licenses_english
9,no_empreendedor,developer_name_original,None


,record_id,element_type_original,element_type_english,sector_original,sector_english
0,560043.0,Área de Porto,Port area,Transporte,Transportation
1,560044.0,Área de Porto,Port area,Transporte,Transportation
2,560045.0,Área de Porto,Port area,Transporte,Transportation
3,560046.0,Área de Porto,Port area,Transporte,Transportation
4,560047.0,Área de Porto,Port area,Transporte,Transportation


## 4. Export verification

The check compares every original attribute and geometry blob with the inventoried source, as well as IDs, row count, CRS, and reviewed translations. It fails before publishing a mismatched file. A licensing entry does not establish that a facility operates or that environmental harm occurred. The frozen website tiles are a separate display output and omit two source IDs; both remain in this research file.

In [5]:
from english_export import verify_export
verify_export(source, result["file"], "ibama", baseline_checksum(ctx.root, source))
assert result["features"] == 8494
print("All 8,494 source records, original values, and geometries are preserved.")

All 8,494 source records, original values, and geometries are preserved.


<details><summary>Execution verification</summary>

The full notebook command completed on 2026-10-27. All 8,494 records matched the preceding reviewed export in IDs, attributes, and geometry blobs.

</details>